# Migration Lab: Traditional GA to Framework GA    传统 GA 到框架 GA

Goal: migrate a classic single-objective GA script into NSGABlack with minimal behavior drift.


目标：把传统单目标 GA 脚本迁移为 NSGABlack 结构，实现行为接近且架构清晰。


In [2]:
from pathlib import Path
import sys

cur = Path.cwd().resolve()
for _ in range(12):
    if (cur / 'pyproject.toml').exists() and (cur / '__init__.py').exists():
        if str(cur.parent) not in sys.path:
            sys.path.insert(0, str(cur.parent))
        print('bootstrap ok:', cur)
        break
    cur = cur.parent


bootstrap ok: C:\Users\hp\Desktop\代码逻辑 - 副本\nsgablack


## Step 0: Baseline responsibilities
第 0 步：传统脚本职责
Traditional scripts usually mix these concerns in one file:

传统脚本通常把以下职责混在一个文件里：
- objective / 目标函数
- init/mutate/repair operators / 初始化-变异-修复算子
- process loop (selection/crossover/update) / 过程循环（选择/交叉/更新）
- runtime control (seed/generations/reporting) / 运行控制（随机种子/代数/输出）


In [3]:
from pathlib import Path
base = Path('00_baseline.py').resolve()
print('baseline script:', base)

baseline script: C:\Users\hp\Desktop\代码逻辑 - 副本\nsgablack\examples\migration_lab\ga_single_objective\00_baseline.py


## Step 1: Move objective into Problem layer / 第 1 步：把目标函数迁入 Problem 层
Create a `BlackBoxProblem` subclass so objective semantics are isolated from GA process logic.

创建 `BlackBoxProblem` 子类，让目标语义与 GA 过程逻辑解耦。


In [4]:
import numpy as np
from nsgablack.core.base import BlackBoxProblem

class ShiftedSphereProblem(BlackBoxProblem):
    def __init__(self, dimension=12, low=-5.0, high=5.0):
        bounds = {f'x{i}': (low, high) for i in range(dimension)}
        super().__init__(name='ShiftedSphere', dimension=dimension, bounds=bounds, objectives=['shifted_sphere'])

    def evaluate(self, x):
        arr = np.asarray(x, dtype=float)
        return float(np.sum((arr - 2.0) ** 2))

## Step 2: Move operators into RepresentationPipeline / 第 2 步：把算子迁入表示管线

Use initializer/mutator/repair components instead of embedding operator logic inside the loop.

使用 initializer/mutator/repair 组件，不要在主循环里硬编码算子逻辑。


In [5]:
from nsgablack.representation import RepresentationPipeline
from nsgablack.representation.continuous import UniformInitializer, GaussianMutation, ClipRepair

pipeline = RepresentationPipeline(
    initializer=UniformInitializer(low=-5.0, high=5.0),
    mutator=GaussianMutation(sigma=0.25, low=-5.0, high=5.0),
    repair=ClipRepair(low=-5.0, high=5.0),
)
pipeline

RepresentationPipeline(encoder=None, repair=ClipRepair(low=-5.0, high=5.0), initializer=UniformInitializer(low=-5.0, high=5.0), initializers=None, max_init_attempts=5, validate_constraints=False, log_validation_failures=False, mutator=GaussianMutation(sigma=0.25, low=-5.0, high=5.0), crossover=None, transactional=False, protect_input=False, copy_context=False, threadsafe=False)

## Step 3: Move process loop into Adapter / 第 3 步：把过程循环迁入 Adapter

Use `NSGA2Adapter` as the process layer (selection + crossover + survivor logic).

使用 `NSGA2Adapter` 作为过程层（选择 + 交叉 + 生存选择逻辑）。


In [6]:
from nsgablack.adapters import NSGA2Adapter, NSGA2Config

adapter = NSGA2Adapter(NSGA2Config(population_size=80, offspring_size=80, crossover_rate=0.9))
adapter

## Step 4: Assemble and run / 第 4 步：组装并运行
`ComposableSolver` is the lifecycle shell: setup/run/context/plugins.

`ComposableSolver` 负责生命周期外壳：setup/run/context/plugins。


In [7]:
from nsgablack.core.composable_solver import ComposableSolver

problem = ShiftedSphereProblem(dimension=12, low=-5.0, high=5.0)
solver = ComposableSolver(problem=problem, adapter=adapter, representation_pipeline=pipeline)
solver.set_max_steps(60)
solver.set_random_seed(7)
result = solver.run()
print(result)
print('best objective:', solver.best_objective)

{'status': 'completed', 'steps': 60, 'steps_executed': 60, 'resume_from': 0, 'elapsed_sec': 20.06081748008728}
best objective: 0.15774930239118753


## Step 5: Compare and verify / 第 5 步：对比与验证
- Baseline file / 传统文件: `00_baseline.py`
- Framework file / 框架文件: `02_framework_final.py`
- Mapping doc / 映射文档: `03_diff.md`

The goal is not exact numeric parity, but architecture-preserving behavior with cleaner extension points.

目标不是数值完全一致，而是在保持行为合理的前提下，把结构迁移到更清晰的可扩展架构。
